# Brain MRI Classification for Dementia Severity Prediction

## Dataset Overview

The dataset consists of brain MRI scans classified into three severity categories:

| Class | Number of Images | Percentage |
|-------|-----------------|------------|
| NoImpairment | 2,560 | 50.4% |
| MildImpairment | 1,792 | 35.3% |
| ModerateImpairment | 724 | 14.3% |
| **Total** | **5,076** | **100%** |

**Image Dimensions:** 176 × 208 pixels (height × width)

**Key Challenge:** Significant class imbalance (ratio ~3.5:1 between largest and smallest class) requires stratified sampling and weighted loss functions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<hr style="border:1px solid black"> </hr>

## 1. Setup and Imports

In [ ]:
# Core Python libraries
import os
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import transforms, datasets
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights

# Stratified splitting for handling class imbalance
from sklearn.model_selection import train_test_split

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
    classification_report,
    auc
)
from sklearn.preprocessing import label_binarize, LabelBinarizer

# Progress bar
from tqdm.notebook import tqdm

# Set matplotlib style
plt.rcParams.update({'font.size': 12})

In [ ]:
# Set random seeds for reproducibility
def set_reproducibility_seed(seed=42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_reproducibility_seed(42)

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

<hr style="border:1px solid black"> </hr>

## 2. Configuration

In [ ]:
# Path - UPDATE FOR YOUR ENVIRONMENT
DATA_DIR = Path('/content/drive/MyDrive/AD')

# Class names (alphabetical - how ImageFolder loads them)
CLASS_NAMES = ['MildImpairment', 'ModerateImpairment', 'NoImpairment']
NUM_CLASSES = 3

# Original MRI dimensions: 176x208 (non-square)
ORIGINAL_HEIGHT = 176
ORIGINAL_WIDTH = 208

# Target size for pretrained models
IMAGE_SIZE = 224

# Padding colour - magenta (255,0,255) contrasts with greyscale brain
DISTINCTIVE_PADDING_COLOUR = (255, 0, 255)

# Training hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

# Stratified split ratios (70% train, 15% val, 15% test)
TEST_SPLIT = 0.15
VAL_SPLIT = 0.15

EARLY_STOPPING_PATIENCE = 5

# Output directory: Output_DD_MM_YYYY__HHMM
TIMESTAMP = datetime.now().strftime('%d_%m_%Y__%H%M')
OUTPUT_DIR = Path(f'./Output_{TIMESTAMP}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for exp_num in range(1, 5):
    (OUTPUT_DIR / f'exp{exp_num}').mkdir(exist_ok=True)

# ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Data: {DATA_DIR}")
print(f"Image: {ORIGINAL_HEIGHT}x{ORIGINAL_WIDTH} -> {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Output: {OUTPUT_DIR}")

<hr style="border:1px solid black"> </hr>

## 3. File Naming Convention

All output files follow a consistent naming convention to easily identify which experiment they belong to:

```
Output_DD_MM_YYYY__HHMM/
├── exp1/                           # Experiment 1: Baseline CNN (no augmentation)
│   ├── Exp1_Baseline_CNN_NoAug_training_history.png
│   ├── Exp1_Baseline_CNN_NoAug_confusion_matrix.png
│   ├── Exp1_Baseline_CNN_NoAug_roc_curves.png
│   ├── Exp1_Baseline_CNN_NoAug_pr_curves.png
│   ├── Exp1_Baseline_CNN_NoAug_prediction_grid.png
│   └── Exp1_Baseline_CNN_NoAug_model.pth
├── exp2/                           # Experiment 2: Baseline CNN + Augmentation
├── exp3/                           # Experiment 3: ResNet50
├── exp4/                           # Experiment 4: ViT-B/16
├── data_splits/                    # Train/Val/Test split indices (for reproducibility)
│   ├── train_indices.npy
│   ├── val_indices.npy
│   └── test_indices.npy
├── Experiment_Comparison_DD_MM_YYYY__HHMM.png       # Final comparison charts
```

**Naming Format:** `Name_DD_MM_YYYY__Time`  
**Example:** `Output_27_01_2026__1037`

<hr style="border:1px solid black"> </hr>

## 4. Custom Transform: Aspect-Ratio-Preserving Resize with Distinctive Padding

In [ ]:
class ResizeWithPad:
    """
    Resize image while preserving aspect ratio, then pad to target size.

    Uses a distinctive padding colour (bright magenta by default) that is very different from grayscale brain tissue to prevent the model from learning patterns from the padded regions.

    Parameters:
        target_size: Target output size (height, width) or single int for square
        fill: RGB tuple for padding colour (default: bright magenta)
    """
    def __init__(self, target_size, fill=DISTINCTIVE_PADDING_COLOUR):
        if isinstance(target_size, int):
            self.target_size = (target_size, target_size)
        else:
            self.target_size = target_size
        self.fill = fill

    def __call__(self, img):
        """
        Resize image preserving aspect ratio and pad with distinctive colour.

        Args:
            img: PIL Image
        Returns:
            PIL Image resized and padded to target_size
        """
        # Get original dimensions
        orig_width, orig_height = img.size
        target_height, target_width = self.target_size

        # Calculate scaling factor to fit within target while preserving aspect ratio
        scale = min(target_width / orig_width, target_height / orig_height)

        # Calculate new dimensions
        new_width = int(orig_width * scale)
        new_height = int(orig_height * scale)

        # Resize image
        img_resized = img.resize((new_width, new_height), Image.BILINEAR)

        # Create new image with padding colour
        new_img = Image.new('RGB', (target_width, target_height), self.fill)

        # Calculate position to paste (centre the image)
        paste_x = (target_width - new_width) // 2
        paste_y = (target_height - new_height) // 2

        # Paste resized image onto padded background
        new_img.paste(img_resized, (paste_x, paste_y))

        return new_img

    def __repr__(self):
        return f"ResizeWithPad(size={self.target_size}, fill=RGB{self.fill})"

print("ResizeWithPad transform defined")
print(f"  - Preserves aspect ratio")
print(f"  - Pads with RGB{DISTINCTIVE_PADDING_COLOUR} (bright magenta)")
print(f"  - Centres the brain image within the padded frame")

In [ ]:
def visualise_padding_transform_comparison(data_dir, output_dir):
    """
    Visualise how the padding transform works and shows original vs. padded images side by side.
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    transform = ResizeWithPad(IMAGE_SIZE, fill=DISTINCTIVE_PADDING_COLOUR)

    # Get sample images from each class
    for idx, class_name in enumerate(CLASS_NAMES):
        class_path = data_dir / class_name
        images = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))
        if images:
            img_path = images[0]
            img_orig = Image.open(img_path).convert('RGB')
            img_padded = transform(img_orig)

            # Original
            axes[0, idx].imshow(img_orig)
            axes[0, idx].set_title(f'{class_name}\nOriginal: {img_orig.size[1]}×{img_orig.size[0]}', fontsize=10)
            axes[0, idx].axis('off')

            # Padded
            axes[1, idx].imshow(img_padded)
            axes[1, idx].set_title(f'Padded: {img_padded.size[1]}×{img_padded.size[0]}\nMagenta padding', fontsize=10)
            axes[1, idx].axis('off')

    # Add explanation in 4th column
    axes[0, 3].text(0.5, 0.5,
                    'Aspect-Ratio\nPreserving\nResize\n\nOriginal:\n176×208',
                    ha='center', va='center', fontsize=14, fontweight='bold')
    axes[0, 3].axis('off')

    axes[1, 3].text(0.5, 0.5,
                    f'Distinctive\nPadding\n\nTarget: {IMAGE_SIZE}×{IMAGE_SIZE}\n\nPad colour:\nRGB{DISTINCTIVE_PADDING_COLOUR}\n(Bright Magenta)',
                    ha='center', va='center', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='#FF00FF', alpha=0.3))
    axes[1, 3].axis('off')

    fig.suptitle('Image Preprocessing: Aspect-Ratio-Preserving Resize with Distinctive Padding\n'
                 '(Bright magenta padding prevents model from learning spurious patterns)',
                 fontsize=14, fontweight='bold')

    plt.tight_layout()
    save_path = output_dir / f'Preprocessing_Demonstration_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

# Will call this after verifying data directory exists

### Every experiment will be evaluated with the exact same metrics and visualisations - accuracy, precision, recall, specificity, F1-score, AUC, confusion matrices, ROC curves, and precision-recall curves. Defining these once and reusing them ensures consistent comparison across all four experiments.

<hr style="border:1px solid black"> </hr>

## 5. Evaluation and Visualisation Helper Functions

In [ ]:
def compute_and_print_classification_metrics(y_true, y_pred, class_names):
    """
    Compute and report classification metrics.
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print("=" * 50)
    print("CLASSIFICATION METRICS")
    print("=" * 50)
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")

    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

In [ ]:
def plot_and_save_confusion_matrix(y_true, y_pred, class_names, exp_name, output_dir):
    """
    Plot and save confusion matrix with proper experiment naming.
    """
    confusion_matrix_array = confusion_matrix(y_true, y_pred)
    matrix_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_array, display_labels=class_names)

    fig, ax = plt.subplots(figsize=(8, 8))
    matrix_display.plot(ax=ax, cmap='Blues', colorbar=True)
    plt.title(f'{exp_name}\nConfusion Matrix', fontweight='bold', fontsize=14)
    plt.xticks(rotation=45, ha='right')

    # Add summary statistics
    total_predictions = confusion_matrix_array.sum()
    correct_predictions = np.trace(confusion_matrix_array)
    overall_accuracy_percentage = correct_predictions / total_predictions * 100
    summary_text = f'Accuracy: {overall_accuracy_percentage:.1f}%\nTotal: {total_predictions} samples'
    plt.gcf().text(0.02, 0.02, summary_text, fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()

    # Save with experiment-specific naming
    save_path = output_dir / f'{exp_name}_Confusion_Matrix_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

    return confusion_matrix_array

In [ ]:
def plot_and_save_roc_curves(y_true, y_prob, class_names, exp_name, output_dir):
    """
    Plot and save ROC curves for multi-class classification.
    """
    number_of_classes = len(class_names)

    # Binarise labels for multi-class ROC
    binarised_true_labels = label_binarize(y_true, classes=range(number_of_classes))

    fig, ax = plt.subplots(figsize=(10, 8))
    class_plot_colours = ['#e74c3c', '#3498db', '#2ecc71']

    # Plot ROC curve for each class
    per_class_auc_scores = []
    for i in range(number_of_classes):
        false_positive_rate, true_positive_rate, _ = roc_curve(binarised_true_labels[:, i], y_prob[:, i])
        class_roc_auc = auc(false_positive_rate, true_positive_rate)
        per_class_auc_scores.append(class_roc_auc)
        ax.plot(false_positive_rate, true_positive_rate, color=class_plot_colours[i], lw=2,
                label=f'{class_names[i]} (AUC = {class_roc_auc:.3f})')

    # Compute macro-average ROC curve
    macro_average_auc = np.mean(per_class_auc_scores)

    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{exp_name}\nROC Curves (Macro AUC = {macro_average_auc:.3f})', fontweight='bold', fontsize=14)
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    save_path = output_dir / f'{exp_name}_ROC_Curves_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

    return macro_average_auc

In [ ]:
def plot_and_save_precision_recall_curves(y_true, y_prob, class_names, exp_name, output_dir):
    """
    Plot and save precision-recall curves for multi-class classification.
    """
    number_of_classes = len(class_names)

    # Binarise labels
    binarised_true_labels = label_binarize(y_true, classes=range(number_of_classes))

    fig, ax = plt.subplots(figsize=(10, 8))
    class_plot_colours = ['#e74c3c', '#3498db', '#2ecc71']

    for i in range(number_of_classes):
        precision_values, recall_values, _ = precision_recall_curve(binarised_true_labels[:, i], y_prob[:, i])
        precision_recall_auc = auc(recall_values, precision_values)
        ax.plot(recall_values, precision_values, color=class_plot_colours[i], lw=2,
                label=f'{class_names[i]} (AUC = {precision_recall_auc:.3f})')

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'{exp_name}\nPrecision-Recall Curves', fontweight='bold', fontsize=14)
    ax.legend(loc='lower left')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    save_path = output_dir / f'{exp_name}_PR_Curves_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
def plot_and_save_training_history(history, exp_name, output_dir):
    """
    Plot and save training history (loss and accuracy curves).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epoch_numbers = range(1, len(history['train_loss']) + 1)

    # Loss plot
    axes[0].plot(epoch_numbers, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(epoch_numbers, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{exp_name}\nTraining and Validation Loss', fontweight='bold', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy plot
    axes[1].plot(epoch_numbers, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(epoch_numbers, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title(f'{exp_name}\nTraining and Validation Accuracy', fontweight='bold', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    save_path = output_dir / f'{exp_name}_Training_History_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
def plot_and_save_prediction_grid(model, test_loader, class_names, exp_name, output_dir, device, num_samples=16):
    """
    Create and save a grid of test predictions with confidence scores.
    """
    model.eval()

    # Collect samples
    collected_images = []
    collected_true_labels = []
    collected_predictions = []
    collected_probabilities = []

    # ImageNet denormalisation for display
    imagenet_mean_tensor = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    imagenet_std_tensor = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    with torch.no_grad():
        for images, labels in test_loader:
            images_on_device = images.to(device)
            outputs = model(images_on_device)
            prediction_probabilities = F.softmax(outputs, dim=1)
            _, batch_predictions = torch.max(outputs, 1)

            for i in range(images.size(0)):
                if len(collected_images) >= num_samples:
                    break
                collected_images.append(images[i])
                collected_true_labels.append(labels[i].item())
                collected_predictions.append(batch_predictions[i].cpu().item())
                collected_probabilities.append(prediction_probabilities[i].cpu().numpy())

            if len(collected_images) >= num_samples:
                break

    # Create grid
    grid_dimension = int(np.ceil(np.sqrt(num_samples)))
    fig, axes = plt.subplots(grid_dimension, grid_dimension, figsize=(4*grid_dimension, 4.5*grid_dimension))
    axes = axes.flatten() if num_samples > 1 else [axes]

    for idx in range(num_samples):
        if idx < len(collected_images):
            # Denormalise image
            denormalised_image = collected_images[idx] * imagenet_std_tensor + imagenet_mean_tensor
            denormalised_image = denormalised_image.permute(1, 2, 0).numpy()
            denormalised_image = np.clip(denormalised_image, 0, 1)

            true_label_index = collected_true_labels[idx]
            predicted_label_index = collected_predictions[idx]
            prediction_confidence = collected_probabilities[idx][predicted_label_index] * 100

            # Determine if correct
            is_prediction_correct = true_label_index == predicted_label_index
            border_colour = 'green' if is_prediction_correct else 'red'

            axes[idx].imshow(denormalised_image)
            axes[idx].set_title(
                f'True: {class_names[true_label_index]}\n'
                f'Pred: {class_names[predicted_label_index]}\n'
                f'Conf: {prediction_confidence:.1f}%',
                fontsize=9,
                color='darkgreen' if is_prediction_correct else 'darkred',
                fontweight='bold'
            )

            # Add coloured border
            for spine in axes[idx].spines.values():
                spine.set_edgecolor(border_colour)
                spine.set_linewidth(3)

            axes[idx].set_xticks([])
            axes[idx].set_yticks([])
        else:
            axes[idx].axis('off')

    # Calculate overall accuracy for this sample
    num_correct_predictions = sum(1 for true, pred in zip(collected_true_labels, collected_predictions) if true == pred)
    sample_accuracy_percentage = num_correct_predictions / len(collected_true_labels) * 100

    fig.suptitle(
        f'{exp_name}\nPrediction Grid ({num_correct_predictions}/{len(collected_true_labels)} correct = {sample_accuracy_percentage:.1f}%)\n'
        f'Green border = Correct | Red border = Incorrect',
        fontweight='bold', fontsize=14, y=1.02
    )

    plt.tight_layout()

    save_path = output_dir / f'{exp_name}_Prediction_Grid_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
def run_complete_evaluation_with_visualisations(model, test_loader, class_names, exp_name, exp_dir, device):
    """
    Run complete evaluation and generate all visualisations for an experiment.

    Returns:
        metrics: Dictionary of evaluation metrics
        auc_score: Macro-averaged AUC score
        y_true, y_pred, y_prob: Arrays for further analysis
    """
    print(f"\n{'='*60}")
    print(f"EVALUATION: {exp_name}")
    print(f"{'='*60}")

    # Get predictions
    model.eval()
    all_true_labels = []
    all_predicted_labels = []
    all_predicted_probabilities = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            prediction_probabilities = F.softmax(outputs, dim=1)
            _, batch_predictions = torch.max(outputs, 1)

            all_true_labels.extend(labels.numpy())
            all_predicted_labels.extend(batch_predictions.cpu().numpy())
            all_predicted_probabilities.extend(prediction_probabilities.cpu().numpy())

    y_true = np.array(all_true_labels)
    y_pred = np.array(all_predicted_labels)
    y_prob = np.array(all_predicted_probabilities)

    # Report metrics
    metrics = compute_and_print_classification_metrics(y_true, y_pred, class_names)

    # Generate and save all visualisations
    print(f"\n--- Generating Visualisations ---")

    # Confusion Matrix
    confusion_matrix_result = plot_and_save_confusion_matrix(y_true, y_pred, class_names, exp_name, exp_dir)

    # ROC Curves
    macro_auc_score = plot_and_save_roc_curves(y_true, y_prob, class_names, exp_name, exp_dir)

    # Precision-Recall Curves
    plot_and_save_precision_recall_curves(y_true, y_prob, class_names, exp_name, exp_dir)

    # Prediction Grid
    plot_and_save_prediction_grid(model, test_loader, class_names, exp_name, exp_dir, device)

    print(f"All visualisations saved to: {exp_dir}")

    return metrics, macro_auc_score, y_true, y_pred, y_prob

<hr style="border:1px solid black"> </hr>

## 6. Single Image Prediction Function

This function allows testing the model on individual images from the test split to verify generalisation.

In [ ]:
def predict_and_visualise_single_image(model, image_path, class_names, device, data_dir=None):
    """
    Make a prediction on a single image and display the result.
    This function allows testing the model on individual images from the test split to verify that the model has truly generalised and is not just memorising the training data.

    Parameters:
        model: Trained PyTorch model
        image_path: Path to the image file (can be full path or just filename)
        class_names: List of class names
        device: torch device
        data_dir: Optional - base data directory to search for the image

    Returns:
        Dictionary with prediction results
    """
    model.eval()

    # Convert to Path object
    image_path = Path(image_path)

    # If just a filename is provided, search for it in the data directory
    if not image_path.exists() and data_dir is not None:
        data_dir = Path(data_dir)
        # Search in all class subdirectories
        for class_name in class_names:
            potential_path = data_dir / class_name / image_path.name
            if potential_path.exists():
                image_path = potential_path
                break

    if not image_path.exists():
        print(f"[Error] Error: Image not found at {image_path}")
        print("Please provide a valid file path or filename.")
        return None

    # Determine true class from directory structure
    true_class = image_path.parent.name if image_path.parent.name in class_names else "Unknown"

    # Load and preprocess image
    original_image = Image.open(image_path).convert('RGB')

    # Apply the same transforms as validation/test
    transform = transforms.Compose([
        ResizeWithPad(IMAGE_SIZE, fill=DISTINCTIVE_PADDING_COLOUR),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    preprocessed_image_tensor = transform(original_image).unsqueeze(0).to(device)

    # Make prediction
    with torch.no_grad():
        outputs = model(preprocessed_image_tensor)
        class_probabilities = F.softmax(outputs, dim=1)[0]
        predicted_class_index = torch.argmax(class_probabilities).item()
        prediction_confidence = class_probabilities[predicted_class_index].item() * 100

    predicted_class = class_names[predicted_class_index]
    is_correct = true_class == predicted_class

    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Original image
    axes[0].imshow(original_image)
    border_colour = 'green' if is_correct else 'red'
    for spine in axes[0].spines.values():
        spine.set_edgecolor(border_colour)
        spine.set_linewidth(4)
    axes[0].set_title(f'Input Image\n{image_path.name}', fontsize=12)
    axes[0].axis('off')

    # Probability bar chart
    bar_colours = ['#3498db' if i != predicted_class_index else '#2ecc71' for i in range(len(class_names))]
    bar_patches = axes[1].barh(class_names, [p.item() * 100 for p in class_probabilities], color=bar_colours, edgecolor='black')
    axes[1].set_xlabel('Confidence (%)', fontsize=12)
    axes[1].set_title('Class Probabilities', fontsize=12, fontweight='bold')
    axes[1].set_xlim(0, 100)

    for bar_patch, class_prob in zip(bar_patches, class_probabilities):
        axes[1].text(bar_patch.get_width() + 1, bar_patch.get_y() + bar_patch.get_height()/2,
                     f'{class_prob.item()*100:.1f}%', va='center', fontsize=11)

    # Add result text
    result_text = f"True Class: {true_class}\nPredicted: {predicted_class}\nConfidence: {prediction_confidence:.1f}%"
    result_colour = 'darkgreen' if is_correct else 'darkred'
    fig.text(0.5, 0.02, result_text, ha='center', fontsize=14, fontweight='bold', color=result_colour,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    status = "* CORRECT" if is_correct else "x INCORRECT"
    fig.suptitle(f'Single Image Prediction Test\n{status}', fontsize=16, fontweight='bold',
                 color=result_colour)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    # Print results
    print("SINGLE IMAGE PREDICTION RESULT")
    print(f"Image: {image_path.name}")
    print(f"True Class: {true_class}")
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {prediction_confidence:.2f}%")
    print(f"Status: {status}")
    print("\nClass Probabilities:")
    for name, class_prob in zip(class_names, class_probabilities):
        print(f"  {name}: {class_prob.item()*100:.2f}%")

    return {
        'image_path': str(image_path),
        'true_class': true_class,
        'predicted_class': predicted_class,
        'confidence': prediction_confidence,
        'is_correct': is_correct,
        'probabilities': {name: class_prob.item() for name, class_prob in zip(class_names, class_probabilities)}
    }

### With the evaluation toolkit in place, we now define how data flows into the models. This covers 3 things: exploring the raw dataset to confirm class distributions and image dimensions, defining data augmentation transforms, and building a stratified splitting pipeline that maintains class proportions across train, validation, and test sets.

<hr style="border:1px solid black"> </hr>

## 7. Data Loading and Exploration

In [ ]:
def explore_dataset_structure_and_distribution(data_dir):
    """
    Explore the dataset structure, class distribution, and image dimensions.
    """
    print("DATASET EXPLORATION")

    class_counts = {}
    total_images = 0
    sample_image_dimensions = None

    for class_name in sorted(os.listdir(data_dir)):
        class_path = data_dir / class_name
        if class_path.is_dir():
            images = list(class_path.glob('*.[jJ][pP][gG]')) + \
                     list(class_path.glob('*.[jJ][pP][eE][gG]')) + \
                     list(class_path.glob('*.[pP][nN][gG]'))
            num_images = len(images)
            class_counts[class_name] = num_images
            total_images += num_images

            if sample_image_dimensions is None and len(images) > 0:
                sample_image = Image.open(images[0])
                sample_image_dimensions = sample_image.size

    print(f"\nDataset Location: {data_dir}")
    print(f"Total images: {total_images}")
    print(f"Number of classes: {len(class_counts)}")
    if sample_image_dimensions:
        print(f"Original image dimensions: {sample_image_dimensions[1]} x {sample_image_dimensions[0]} (H x W)")

    print(f"\nClass distribution:")
    for class_name, count in sorted(class_counts.items()):
        percentage = (count / total_images) * 100
        bar = '█' * int(percentage / 2)
        print(f"  {class_name:20s}: {count:>5} images ({percentage:>5.1f}%) {bar}")

    count_values = list(class_counts.values())
    class_imbalance_ratio = max(count_values) / min(count_values)
    print(f"\nClass imbalance ratio: {class_imbalance_ratio:.2f}:1")
    if class_imbalance_ratio > 2:
      return class_counts, total_images

# Check if data directory exists
if DATA_DIR.exists():
    class_counts, total_images = explore_dataset_structure_and_distribution(DATA_DIR)
    # Demonstrate padding transform
    visualise_padding_transform_comparison(DATA_DIR, OUTPUT_DIR)
else:
    print(f"Data directory not found: {DATA_DIR}")
    print("Please update DATA_DIR in the Configuration section.")
    class_counts, total_images = {}, 0

In [ ]:
def plot_and_save_class_distribution_chart(class_counts, output_dir):
    """Visualise class distribution as a bar chart."""
    if not class_counts:
        print("No class data to visualise.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    class_name_labels = list(class_counts.keys())
    image_counts_per_class = list(class_counts.values())
    total_image_count = sum(image_counts_per_class)
    bar_colours = ['#2ecc71', '#e74c3c', '#3498db']

    bar_patches = ax.bar(class_name_labels, image_counts_per_class, color=bar_colours, edgecolor='black', linewidth=1.5)

    for bar_patch, count in zip(bar_patches, image_counts_per_class):
        percentage = count / total_image_count * 100
        ax.text(bar_patch.get_x() + bar_patch.get_width()/2, bar_patch.get_height() + 30,
                f'{count}\n({percentage:.1f}%)', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    ax.set_xlabel('Dementia Severity Class', fontsize=12)
    ax.set_ylabel('Number of Images', fontsize=12)
    ax.set_title('Class Distribution in Brain MRI Dataset\n(Significant Imbalance Present)',
                 fontweight='bold', fontsize=14)
    ax.set_ylim(0, max(image_counts_per_class) * 1.15)

    class_imbalance_ratio = max(image_counts_per_class) / min(image_counts_per_class)

    # Moved to top-LEFT corner to avoid overlapping with NoImpairment bar
    ax.text(0.02, 0.95, f'Imbalance Ratio: {class_imbalance_ratio:.1f}:1',
            transform=ax.transAxes, ha='left', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
            fontsize=11, fontweight='bold')

    plt.xticks(rotation=15)
    plt.tight_layout()

    save_path = output_dir / f'Class_Distribution_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

if class_counts:
    plot_and_save_class_distribution_chart(class_counts, OUTPUT_DIR)

<hr style="border:1px solid black"> </hr>

## 8. Data Preparation with Stratified Splitting

**Important:** If you re-run the data splitting cells, the old split folders will be automatically replaced with new ones to avoid duplicate training datasets. 
The cell below defines two transform pipelines - one with no augmentation (used as the control condition in Experiments 1, 3 and 4) and one with clinically appropriate augmentations (used in Experiments 2).

In [ ]:
# Basic transforms (no augmentation) - Exp 1, 3 and 4
baseline_transforms_no_augmentation = transforms.Compose([
    ResizeWithPad(IMAGE_SIZE, fill=DISTINCTIVE_PADDING_COLOUR),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Augmentation transforms - Exp 2
training_augmentation_transforms = transforms.Compose([
    ResizeWithPad(IMAGE_SIZE, fill=DISTINCTIVE_PADDING_COLOUR),
    transforms.RandomAffine(
        degrees=0,              # No rotation
        translate=(0.10, 0.10), # Patient positioning variation
        scale=(0.90, 1.10),     # Field-of-view differences
        fill=DISTINCTIVE_PADDING_COLOUR
    ),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), ratio=(0.5, 2.0))
])

# Validation/Test transforms (no augmentation)
validation_and_test_transforms = transforms.Compose([
    ResizeWithPad(IMAGE_SIZE, fill=DISTINCTIVE_PADDING_COLOUR),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print(f"Transforms: {ORIGINAL_HEIGHT}x{ORIGINAL_WIDTH} -> {IMAGE_SIZE}x{IMAGE_SIZE}")
print("baseline_transforms_no_augmentation: resize + normalise")
print("training_augmentation_transforms: resize + affine + jitter + blur + erasing + normalise")
print("validation_and_test_transforms: resize + normalise")

In [ ]:
class TransformableDataSubset(Dataset):
    """
    Wrapper to apply transforms to a subset of data and allows different transforms for train/val/test splits.
    """
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Global variable to store test indices for single image testing
GLOBAL_TEST_SET_INDICES = None
GLOBAL_COMPLETE_DATASET = None

def create_stratified_train_validation_test_loaders(data_dir, train_transforms, validation_and_test_transforms, batch_size,
                                    test_split=0.15, val_split=0.15, use_weighted_sampling=True,
                                    save_splits=True, output_dir=None):
    """
    Create train, validation, and test data loaders with STRATIFIED splitting.

    If save_splits is True, the split indices will be saved to disk.
    If the splits directory already exists, it will be REPLACED with new splits.
    """
    global GLOBAL_TEST_SET_INDICES, GLOBAL_COMPLETE_DATASET

    print("CREATING STRATIFIED DATA SPLITS")

    # Handle existing splits directory - REPLACE if exists
    if output_dir is not None and save_splits:
        splits_dir = Path(output_dir) / 'data_splits'
        if splits_dir.exists():
            print(f"\nExisting splits directory found at {splits_dir}")
            shutil.rmtree(splits_dir)
        splits_dir.mkdir(parents=True, exist_ok=True)

    full_dataset = datasets.ImageFolder(root=data_dir)
    GLOBAL_COMPLETE_DATASET = full_dataset  # Store for single image testing
    class_names = full_dataset.classes
    targets = np.array(full_dataset.targets)
    indices = np.arange(len(full_dataset))

    print(f"\nTotal samples: {len(full_dataset)}")
    print(f"Classes: {class_names}")

    # First split: separate test set (stratified)
    train_val_indices, test_indices = train_test_split(
        indices,
        test_size=test_split,
        stratify=targets,
        random_state=42
    )

    # Second split: separate validation from training (stratified)
    train_val_targets = targets[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=val_split / (1 - test_split),
        stratify=train_val_targets,
        random_state=42
    )

    # Store test indices globally for single image testing
    GLOBAL_TEST_SET_INDICES = test_indices

    print(f"\nStratified split sizes:")
    print(f"  Training:   {len(train_indices)} samples ({len(train_indices)/len(full_dataset)*100:.1f}%)")
    print(f"  Validation: {len(val_indices)} samples ({len(val_indices)/len(full_dataset)*100:.1f}%)")
    print(f"  Test:       {len(test_indices)} samples ({len(test_indices)/len(full_dataset)*100:.1f}%)")

    # Save splits if requested
    if output_dir is not None and save_splits:
        np.save(splits_dir / 'train_indices.npy', train_indices)
        np.save(splits_dir / 'val_indices.npy', val_indices)
        np.save(splits_dir / 'test_indices.npy', test_indices)
        print(f"Split indices saved to: {splits_dir}")

        # Also save the file paths for the test set (useful for single image testing)
        test_files = [full_dataset.samples[i][0] for i in test_indices]
        with open(splits_dir / 'test_files.txt', 'w') as f:
            for file_path in test_files:
                f.write(f"{file_path}\n")
        print(f"Test file paths saved to: {splits_dir / 'test_files.txt'}")

    # Verify stratification
    print(f"\nClass distribution verification:")
    for split_name, split_indices in [('Train', train_indices), ('Val', val_indices), ('Test', test_indices)]:
        split_targets = targets[split_indices]
        counts = Counter(split_targets)
        total = len(split_indices)
        props = [f"{class_names[i]}: {counts[i]/total*100:.1f}%" for i in range(len(class_names))]
        print(f"  {split_name}: {', '.join(props)}")

    # Create subsets
    train_subset = Subset(full_dataset, train_indices)
    val_subset = Subset(full_dataset, val_indices)
    test_subset = Subset(full_dataset, test_indices)

    # Calculate class weights
    train_targets = targets[train_indices]
    class_counts = Counter(train_targets)
    total_train = len(train_targets)

    class_weights = torch.tensor([total_train / (len(class_names) * class_counts[i])
                                   for i in range(len(class_names))], dtype=torch.float32)

    print(f"\nClass weights for loss function (inverse frequency):")
    for i, name in enumerate(class_names):
        print(f"  {name}: {class_weights[i]:.4f}")

    # Apply transforms
    train_dataset = TransformableDataSubset(train_subset, train_transforms)
    val_dataset = TransformableDataSubset(val_subset, validation_and_test_transforms)
    test_dataset = TransformableDataSubset(test_subset, validation_and_test_transforms)

    # Create weighted sampler
    if use_weighted_sampling:
        sample_weights = [class_weights[targets[i]].item() for i in train_indices]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                  sampler=sampler, num_workers=2, pin_memory=True)
        print(f"Using weighted random sampling for training (addresses class imbalance)")
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                  shuffle=True, num_workers=2, pin_memory=True)

    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader, class_names, class_weights

In [ ]:
# Global variables for test set access
GLOBAL_TEST_SET_INDICES = None
GLOBAL_COMPLETE_DATASET = None

# Class prefix mapping (lowercase for matching)
LOWERCASE_TO_FOLDER_NAME_MAP = {
    'noimpairment': 'NoImpairment',
    'mildimpairment': 'MildImpairment',
    'moderateimpairment': 'ModerateImpairment'
}

def print_test_set_class_statistics():
    """
    Print statistics about the test set.
    These images were never used during training.
    """
    global GLOBAL_TEST_SET_INDICES, GLOBAL_COMPLETE_DATASET

    if GLOBAL_TEST_SET_INDICES is None or GLOBAL_COMPLETE_DATASET is None:
        print("Error: Data splits have not been created yet.")
        return None

    # Count images per class
    class_counts = {}
    for idx in GLOBAL_TEST_SET_INDICES:
        file_path = GLOBAL_COMPLETE_DATASET.samples[idx][0]
        class_name = Path(file_path).parent.name
        class_counts[class_name] = class_counts.get(class_name, 0) + 1

    print(f"Test set: {len(GLOBAL_TEST_SET_INDICES)} images (never used in training)")
    for class_name in sorted(class_counts.keys()):
        count = class_counts[class_name]
        pct = count / len(GLOBAL_TEST_SET_INDICES) * 100
        print(f"  {class_name}: {count} ({pct:.1f}%)")

    return class_counts

def select_random_test_image_path(random_class=True, target_class=None):
    """
    Get a random image path from the test set.

    Parameters:
        random_class: If True, select from any class. If False, filter by target_class.
        target_class: Class to filter by ('noimpairment', 'mildimpairment', 'moderateimpairment')

    Returns:
        Path to the selected test image
    """
    global GLOBAL_TEST_SET_INDICES, GLOBAL_COMPLETE_DATASET

    if GLOBAL_TEST_SET_INDICES is None or GLOBAL_COMPLETE_DATASET is None:
        print("Error: Data splits have not been created yet.")
        return None

    if random_class:
        random_idx = random.choice(GLOBAL_TEST_SET_INDICES)
        return GLOBAL_COMPLETE_DATASET.samples[random_idx][0]

    # Filter by target class
    if target_class is None:
        print("Error: target_class required when random_class=False")
        print("Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'")
        return None

    target_class_lower = target_class.lower().strip()

    if target_class_lower not in LOWERCASE_TO_FOLDER_NAME_MAP:
        print(f"Error: Invalid target_class '{target_class}'")
        print("Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'")
        return None

    class_folder_name = LOWERCASE_TO_FOLDER_NAME_MAP[target_class_lower]

    # Filter test indices to target class
    filtered_indices = [
        idx for idx in GLOBAL_TEST_SET_INDICES
        if Path(GLOBAL_COMPLETE_DATASET.samples[idx][0]).parent.name == class_folder_name
    ]

    if len(filtered_indices) == 0:
        print(f"Error: No test images found for class '{target_class}'")
        return None

    random_idx = random.choice(filtered_indices)
    return GLOBAL_COMPLETE_DATASET.samples[random_idx][0]

def list_random_test_image_paths(n=10, target_class=None):
    """
    List n random image paths from the test set.

    Parameters:
        n: Number of images to list
        target_class: Optional class filter
    """
    global GLOBAL_TEST_SET_INDICES, GLOBAL_COMPLETE_DATASET

    if GLOBAL_TEST_SET_INDICES is None or GLOBAL_COMPLETE_DATASET is None:
        print("Error: Data splits have not been created yet.")
        return []

    indices_to_sample = list(GLOBAL_TEST_SET_INDICES)

    if target_class is not None:
        target_class_lower = target_class.lower().strip()
        if target_class_lower in LOWERCASE_TO_FOLDER_NAME_MAP:
            class_folder_name = LOWERCASE_TO_FOLDER_NAME_MAP[target_class_lower]
            indices_to_sample = [
                idx for idx in GLOBAL_TEST_SET_INDICES
                if Path(GLOBAL_COMPLETE_DATASET.samples[idx][0]).parent.name == class_folder_name
            ]

    sample_indices = random.sample(indices_to_sample, min(n, len(indices_to_sample)))
    paths = [GLOBAL_COMPLETE_DATASET.samples[i][0] for i in sample_indices]

    for i, path in enumerate(paths, 1):
        class_name = Path(path).parent.name
        filename = Path(path).name
        print(f"{i}. [{class_name}] {filename}")

    return paths

### The data pipeline is ready. Now we define the 4 model architectures. Each architecture is a self-contained class and the experiment sections later will simply instantiate them with the appropriate settings.

<hr style="border:1px solid black"> </hr>

## 9. Model Definitions

In [ ]:
class BaselineCNN(nn.Module):
    """
    Simple CNN from scratch - baseline model.

    Architecture Design Choices:
    - Uses Global Average Pooling (GAP) instead of large fully-connected layers
    - This reduces parameters to prevent memorisation
    - GAP forces the network to learn spatially-invariant features
    - Batch normalisation for training stability
    - Dropout for regularisation

    The small parameter count ensures the model must learn generalisable features rather than memorising the training set.
    """
    def __init__(self, num_classes, input_size=224):
        super(BaselineCNN, self).__init__()

        # Feature extraction with batch normalisation for stability
        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),

            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),

            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),

            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),
        )

        # Global Average Pooling - reduces 256x14x14 to 256x1x1
        # This is KEY: prevents memorisation by removing spatial information
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Small classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

        self._print_parameter_count_summary()

    def _print_parameter_count_summary(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\nBaseline CNN Parameters:")
        print(f"  Total:     {total_params:,}")
        print(f"  Trainable: {trainable_params:,}")
        print(f"  Architecture: 4 conv blocks + Global Average Pooling + small MLP")

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

In [ ]:
class ImprovedBaselineCNN(nn.Module):
    """
    Improved CNN from scratch - designed to better fit the training data.

    Key Changes from BaselineCNN:
    - REDUCED dropout (0.1 vs 0.25 in conv blocks, 0.25 vs 0.5 in classifier)
    - LARGER classifier head (256 hidden units vs 128)
    - The original model was underfitting so we reduce regularisation
    """
    def __init__(self, num_classes, input_size=224):
        super(ImprovedBaselineCNN, self).__init__()

        # Feature extraction with REDUCED dropout (was 0.25, now 0.1)
        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1),  # REDUCED from 0.25

            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1),  # REDUCED from 0.25

            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1),  # REDUCED from 0.25

            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1),  # REDUCED from 0.25
        )

        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # LARGER classifier head with REDUCED dropout
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),      # INCREASED from 128
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),         # REDUCED from 0.5
            nn.Linear(256, num_classes)
        )

        self._print_parameter_count_summary()

    def _print_parameter_count_summary(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\nImproved Baseline CNN Parameters:")
        print(f"  Total:     {total_params:,}")
        print(f"  Trainable: {trainable_params:,}")
        print(f"  Architecture: 4 conv blocks + GAP + larger MLP")
        print(f"  Key changes: Reduced dropout (0.1 conv, 0.25 fc), larger classifier")

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

In [ ]:
class ResNet50Classifier(nn.Module):
    """
    ResNet50 with transfer learning and configurable fine-tuning.

    Fine-tuning modes:
    - 'none': Freeze all backbone weights (only train classifier head)
    - 'partial': Unfreeze layer3 and layer4 (recommended for domain adaptation)
    - 'full': Unfreeze entire backbone (risk of catastrophic forgetting)

    For medical imaging, 'partial' is recommended because:
    - Early layers (layer1, layer2) learn generic features (edges, textures)
    - Later layers (layer3, layer4) learn domain-specific features
    - Unfreezing later layers allows adaptation to MRI characteristics
    """
    def __init__(self, num_classes, fine_tune_mode='partial'):
        super(ResNet50Classifier, self).__init__()

        print("Loading ResNet50 backbone (ImageNet pretrained)...")
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        # Apply freezing strategy based on mode
        if fine_tune_mode == 'none':
            print("Freezing ALL backbone weights (classifier head only)")
            for param in self.backbone.parameters():
                param.requires_grad = False
        elif fine_tune_mode == 'partial':
            print("Partial fine-tuning: freezing layer1, layer2; unfreezing layer3, layer4")
            # Freeze early layers
            for name, param in self.backbone.named_parameters():
                if 'layer1' in name or 'layer2' in name or 'conv1' in name or 'bn1' in name:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
        else:  # 'full'
            print("Full fine-tuning: all backbone weights trainable")
            for param in self.backbone.parameters():
                param.requires_grad = True

        # Replace classifier head
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        self._print_parameter_count_summary()

    def _print_parameter_count_summary(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen_params = total_params - trainable_params
        print(f"\nResNet50 Parameters:")
        print(f"  Total:     {total_params:,}")
        print(f"  Trainable: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
        print(f"  Frozen:    {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")

    def forward(self, x):
        return self.backbone(x)

In [ ]:
class ViTClassifier(nn.Module):
    """
    Vision Transformer (ViT-B/16) classifier using torchvision.

    Fine-tuning modes:
    - 'none': Freeze all backbone weights (only train classifier head)
    - 'partial': Unfreeze last N transformer layers (recommended)
    - 'full': Unfreeze entire backbone

    IMPROVED: For medical imaging with small datasets:
    - Unfreeze MORE layers (8-10 instead of 4)
    - Unfreeze the PATCH EMBEDDING layer - critical for domain shift to grayscale MRI
    - Use lower learning rate to prevent catastrophic forgetting
    """
    def __init__(self, num_classes, fine_tune_mode='partial', num_unfrozen_layers=8,
                 unfreeze_patch_embed=True, dropout_rate=0.3):
        super(ViTClassifier, self).__init__()

        print("Loading ViT-B/16 backbone (ImageNet-1K pretrained)...")
        print("  - Architecture: Vision Transformer Base with 16x16 patches")
        print("  - Parameters: ~86M total (768-dim embeddings, 12 layers, 12 heads)")

        weights = ViT_B_16_Weights.IMAGENET1K_V1
        self.backbone = vit_b_16(weights=weights)

        self.embedding_dim = self.backbone.heads.head.in_features
        print(f"  - Embedding dimension: {self.embedding_dim}")

        # Apply freezing strategy based on mode
        if fine_tune_mode == 'none':
            print("Freezing ALL backbone weights (classifier head only)")
            for param in self.backbone.parameters():
                param.requires_grad = False
        elif fine_tune_mode == 'partial':
            print(f"Partial fine-tuning: unfreezing last {num_unfrozen_layers} transformer layers")
            # First freeze everything
            for param in self.backbone.parameters():
                param.requires_grad = False

            # Unfreeze last N encoder layers (INCREASED from 4 to 8)
            encoder_layers = list(self.backbone.encoder.layers)
            for layer in encoder_layers[-num_unfrozen_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True

            # Unfreeze the encoder layer norm
            for param in self.backbone.encoder.ln.parameters():
                param.requires_grad = True

            # Unfreeze patch embedding for domain adaptation to medical imaging
            if unfreeze_patch_embed:
                print("  - ALSO unfreezing patch embedding (critical for MRI domain shift)")
                # Unfreeze conv_proj (the patch embedding convolution)
                for param in self.backbone.conv_proj.parameters():
                    param.requires_grad = True
                # Unfreeze class token and position embeddings
                if hasattr(self.backbone, 'class_token'):
                    self.backbone.class_token.requires_grad = True
                # Position embeddings
                if hasattr(self.backbone, 'encoder') and hasattr(self.backbone.encoder, 'pos_embedding'):
                    self.backbone.encoder.pos_embedding.requires_grad = True

        else:  # 'full'
            print("Full fine-tuning: all backbone weights trainable")
            for param in self.backbone.parameters():
                param.requires_grad = True

        # Replace classifier head
        self.backbone.heads.head = nn.Sequential(
            nn.LayerNorm(self.embedding_dim),
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.embedding_dim, 256),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )

        # Always ensure classifier head is trainable
        for param in self.backbone.heads.parameters():
            param.requires_grad = True

        self._print_parameter_count_summary()

    def _print_parameter_count_summary(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen_params = total_params - trainable_params
        print(f"\nViT-B/16 Parameters:")
        print(f"  Total:     {total_params:,}")
        print(f"  Trainable: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
        print(f"  Frozen:    {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")

    def forward(self, x):
        return self.backbone(x)

### The final piece of shared infrastructure is the training loop. All 4 experiments use the same `train_model_with_early_stopping` function, which incorporates early stopping to prevent overfitting and logs training/validation metrics per epoch. This means any performance differences between experiments are attributable to the model choices and not to differences in how training was conducted.

<hr style="border:1px solid black"> </hr>

## 10. Training Functions

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        else:
            self.best_loss = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        return False

    def restore(self, model):
        if self.restore_best_weights and self.best_weights is not None:
            model.load_state_dict(self.best_weights)

In [ ]:
def train_model_for_single_epoch(model, train_loader, criterion, optimiser, device):
    """Train the model for one epoch."""
    model.train()
    cumulative_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in tqdm(train_loader, desc='Training', leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        cumulative_loss += loss.item() * images.size(0)
        _, predicted_classes = torch.max(outputs, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted_classes == labels).sum().item()

    epoch_loss = cumulative_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    cumulative_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            cumulative_loss += loss.item() * images.size(0)
            _, predicted_classes = torch.max(outputs, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted_classes == labels).sum().item()

    epoch_loss = cumulative_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
def train_model_with_early_stopping(model, train_loader, val_loader, criterion, optimiser,
                num_epochs, device, patience=5, exp_name='model', exp_dir=None):
    """
    Full training loop with early stopping and automatic visualisation saving.
    """
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    early_stopping = EarlyStopping(patience=patience, restore_best_weights=True)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    for epoch in range(num_epochs):
        training_epoch_loss, training_epoch_accuracy = train_model_for_single_epoch(model, train_loader, criterion, optimiser, device) #calling the train_model_for_single_epoch function
        validation_epoch_loss, validation_epoch_accuracy = validate(model, val_loader, criterion, device)

        history['train_loss'].append(training_epoch_loss)
        history['train_acc'].append(training_epoch_accuracy)
        history['val_loss'].append(validation_epoch_loss)
        history['val_acc'].append(validation_epoch_accuracy)

        print(f"Epoch {epoch+1:02d}/{num_epochs}: "
              f"Train Loss: {training_epoch_loss:.4f}, Train Acc: {training_epoch_accuracy:.4f} | "
              f"Val Loss: {validation_epoch_loss:.4f}, Val Acc: {validation_epoch_accuracy:.4f}")

        if early_stopping(validation_epoch_loss, model):
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            early_stopping.restore(model)
            break

    print(f"\nTraining completed. Best validation loss: {early_stopping.best_loss:.4f}")

    # Save training history plot
    if exp_dir:
        plot_and_save_training_history(history, exp_name, exp_dir)

    return history

<hr style="border:1px solid black"> </hr>

## 11. Experiment 1: Baseline CNN (No Augmentation)

This establishes our baseline performance using a simple CNN trained from scratch **without** any data augmentation.

In [ ]:
print("EXPERIMENT 1: BASELINE CNN (NO AUGMENTATION)")

EXPERIMENT_1_NAME = "Exp1_Baseline_CNN_NoAug"
EXPERIMENT_1_OUTPUT_DIR = OUTPUT_DIR / 'exp1'

# Create data loaders with NO augmentation
training_loader_experiment1, validation_loader_experiment1, test_loader_experiment1, class_names, class_weights = create_stratified_train_validation_test_loaders(
    DATA_DIR,
    train_transforms=baseline_transforms_no_augmentation,
    validation_and_test_transforms=validation_and_test_transforms,
    batch_size=BATCH_SIZE,
    test_split=TEST_SPLIT,
    val_split=VAL_SPLIT,
    save_splits=True,
    output_dir=OUTPUT_DIR
)

In [ ]:
# Create baseline model
set_reproducibility_seed(42)
baseline_cnn_model = BaselineCNN(num_classes=NUM_CLASSES, input_size=IMAGE_SIZE).to(device)

baseline_cnn_loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))
baseline_cnn_optimiser = optim.Adam(baseline_cnn_model.parameters(), lr=LEARNING_RATE)

print(f"\nLoss: CrossEntropyLoss (with class weights for imbalance)")
print(f"Optimiser: Adam (lr={LEARNING_RATE})")

In [ ]:
# Train baseline model
baseline_cnn_training_history = train_model_with_early_stopping(
    baseline_cnn_model, training_loader_experiment1, validation_loader_experiment1, baseline_cnn_loss_function, baseline_cnn_optimiser,
    num_epochs=NUM_EPOCHS, device=device, patience=EARLY_STOPPING_PATIENCE,
    exp_name=EXPERIMENT_1_NAME, exp_dir=EXPERIMENT_1_OUTPUT_DIR
)

In [ ]:
# Run full evaluation and generate all visualisations
baseline_cnn_metrics, baseline_cnn_auc_score, baseline_cnn_true_labels, baseline_cnn_predicted_labels, baseline_cnn_predicted_probabilities = run_complete_evaluation_with_visualisations(
    baseline_cnn_model, test_loader_experiment1, class_names, EXPERIMENT_1_NAME, EXPERIMENT_1_OUTPUT_DIR, device
)

# Save model
torch.save(baseline_cnn_model.state_dict(), EXPERIMENT_1_OUTPUT_DIR / f'{EXPERIMENT_1_NAME}_Model_{TIMESTAMP}.pth')
print(f"\nModel saved: {EXPERIMENT_1_OUTPUT_DIR / f'{EXPERIMENT_1_NAME}_Model_{TIMESTAMP}.pth'}")

### Test on Unseen Images (Experiment 1)

Test the trained model on individual images from the **test set** — images that were **never seen during training**. This demonstrates whether the model has truly learned to generalise rather than memorising training data.

**Testing Options:**
- Set `RANDOM_CLASS = True` for fully random test image selection
- Set `RANDOM_CLASS = False` and specify `TARGET_CLASS` to test specific classes (useful for investigating confusion matrix patterns)

In [ ]:
# Test set statistics
print_test_set_class_statistics()

In [ ]:
# Test on unseen image from test set
RANDOM_CLASS = True
TARGET_CLASS = 'mildimpairment'  # Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'

SELECTED_TEST_IMAGE_PATH = select_random_test_image_path(
    random_class=RANDOM_CLASS,
    target_class=TARGET_CLASS if not RANDOM_CLASS else None
)

if SELECTED_TEST_IMAGE_PATH:
    baseline_cnn_single_image_result = predict_and_visualise_single_image(
        model=baseline_cnn_model,
        image_path=SELECTED_TEST_IMAGE_PATH,
        class_names=class_names,
        device=device,
        data_dir=DATA_DIR
    )

<hr style="border:1px solid black"> </hr>

## 12. Experiment 2: Baseline CNN with Data Augmentation

Now we evaluate the impact of data augmentation on the same baseline architecture.

In [ ]:
print("EXPERIMENT 2: BASELINE CNN WITH AUGMENTATION")

EXPERIMENT_2_NAME = "Exp2_Improved_Baseline_CNN_WithAug"
EXPERIMENT_2_OUTPUT_DIR = OUTPUT_DIR / 'exp2'

# Improvements over Experiment 1:
# - Reduced dropout (0.1 conv, 0.25 fc) to address underfitting
# - Larger classifier head (256 vs 128 hidden units)
# - More aggressive augmentation

training_loader_experiment2, validation_loader_experiment2, test_loader_experiment2, _, _ = create_stratified_train_validation_test_loaders(
    DATA_DIR,
    train_transforms=training_augmentation_transforms,
    validation_and_test_transforms=validation_and_test_transforms,
    batch_size=BATCH_SIZE,
    test_split=TEST_SPLIT,
    val_split=VAL_SPLIT,
    save_splits=False,
    output_dir=OUTPUT_DIR
)

In [ ]:
# Create IMPROVED baseline model with reduced regularisation
set_reproducibility_seed(42)
improved_baseline_cnn_model = ImprovedBaselineCNN(num_classes=NUM_CLASSES, input_size=IMAGE_SIZE).to(device)

# Use HIGHER learning rate (5x the original) to help with underfitting
EXPERIMENT_2_LEARNING_RATE = 5e-4  # Increased from 1e-4

improved_baseline_loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))
improved_baseline_optimiser = optim.Adam(improved_baseline_cnn_model.parameters(), lr=EXPERIMENT_2_LEARNING_RATE)

print(f"\nLoss: CrossEntropyLoss (with class weights for imbalance)")
print(f"Optimiser: Adam (lr={EXPERIMENT_2_LEARNING_RATE}) - 5x higher than baseline")
print(f"\nKey improvements over Experiment 1:")
print(f"  - Reduced dropout: 0.1 (conv) + 0.25 (fc) vs 0.25 + 0.5")
print(f"  - Larger classifier: 256 hidden units vs 128")
print(f"  - Higher learning rate: {EXPERIMENT_2_LEARNING_RATE} vs {LEARNING_RATE}")
print(f"  - More aggressive augmentation (see above)")

In [ ]:
# Train with augmentation
improved_baseline_training_history = train_model_with_early_stopping(
    improved_baseline_cnn_model, training_loader_experiment2, validation_loader_experiment2, improved_baseline_loss_function, improved_baseline_optimiser,
    num_epochs=NUM_EPOCHS, device=device, patience=EARLY_STOPPING_PATIENCE,
    exp_name=EXPERIMENT_2_NAME, exp_dir=EXPERIMENT_2_OUTPUT_DIR
)

In [ ]:
# Run full evaluation
improved_baseline_metrics, improved_baseline_auc_score, improved_baseline_true_labels, improved_baseline_predicted_labels, improved_baseline_predicted_probabilities = run_complete_evaluation_with_visualisations(
    improved_baseline_cnn_model, test_loader_experiment2, class_names, EXPERIMENT_2_NAME, EXPERIMENT_2_OUTPUT_DIR, device
)

# Save model
torch.save(improved_baseline_cnn_model.state_dict(), EXPERIMENT_2_OUTPUT_DIR / f'{EXPERIMENT_2_NAME}_Model_{TIMESTAMP}.pth')
print(f"\nModel saved: {EXPERIMENT_2_OUTPUT_DIR / f'{EXPERIMENT_2_NAME}_Model_{TIMESTAMP}.pth'}")

### Test on Unseen Images (Experiment 2)

Test the trained model on individual images from the **test set** — images that were **never seen during training**. This demonstrates whether the model has truly learned to generalise rather than memorising training data.

**Testing Options:**
- Set `RANDOM_CLASS = True` for fully random test image selection
- Set `RANDOM_CLASS = False` and specify `TARGET_CLASS` to test specific classes (useful for investigating confusion matrix patterns)

In [ ]:
# Test set statistics
print_test_set_class_statistics()

In [ ]:
# Test on unseen image from test set
RANDOM_CLASS = True
TARGET_CLASS = 'mildimpairment'  # Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'

SELECTED_TEST_IMAGE_PATH = select_random_test_image_path(
    random_class=RANDOM_CLASS,
    target_class=TARGET_CLASS if not RANDOM_CLASS else None
)

if SELECTED_TEST_IMAGE_PATH:
    improved_baseline_single_image_result = predict_and_visualise_single_image(
        model=improved_baseline_cnn_model,
        image_path=SELECTED_TEST_IMAGE_PATH,
        class_names=class_names,
        device=device,
        data_dir=DATA_DIR
    )

<hr style="border:1px solid black"> </hr>

## 13. Experiment 3: Transfer Learning with ResNet50

In [ ]:
print("EXPERIMENT 3: TRANSFER LEARNING (RESNET50)")

EXPERIMENT_3_NAME = "Exp3_ResNet50_TransferLearning"
EXPERIMENT_3_OUTPUT_DIR = OUTPUT_DIR / 'exp3'

# NO augmentation - isolate transfer learning as the only variable
training_loader_experiment3, validation_loader_experiment3, test_loader_experiment3, _, _ = create_stratified_train_validation_test_loaders(
    DATA_DIR,
    train_transforms=baseline_transforms_no_augmentation,
    validation_and_test_transforms=validation_and_test_transforms,
    batch_size=BATCH_SIZE,
    test_split=TEST_SPLIT,
    val_split=VAL_SPLIT,
    save_splits=False,
    output_dir=OUTPUT_DIR
)

In [ ]:
# Create ResNet50 model with PARTIAL fine-tuning
# This allows the model to adapt later layers to medical imaging domain
# while keeping early generic feature detectors frozen
set_reproducibility_seed(42)
resnet50_transfer_learning_model = ResNet50Classifier(num_classes=NUM_CLASSES, fine_tune_mode='partial').to(device)

resnet50_loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))
# Note: We use a lower learning rate for fine-tuning pretrained weights
resnet50_optimiser = optim.Adam(filter(lambda p: p.requires_grad, resnet50_transfer_learning_model.parameters()), lr=LEARNING_RATE * 0.1)

print(f"\nUsing lower learning rate for fine-tuning: {LEARNING_RATE * 0.1}")

In [ ]:
# Train ResNet50
resnet50_training_history = train_model_with_early_stopping(
    resnet50_transfer_learning_model, training_loader_experiment3, validation_loader_experiment3, resnet50_loss_function, resnet50_optimiser,
    num_epochs=NUM_EPOCHS, device=device, patience=EARLY_STOPPING_PATIENCE,
    exp_name=EXPERIMENT_3_NAME, exp_dir=EXPERIMENT_3_OUTPUT_DIR
)

In [ ]:
# Run full evaluation
resnet50_metrics, resnet50_auc_score, resnet50_true_labels, resnet50_predicted_labels, resnet50_predicted_probabilities = run_complete_evaluation_with_visualisations(
    resnet50_transfer_learning_model, test_loader_experiment3, class_names, EXPERIMENT_3_NAME, EXPERIMENT_3_OUTPUT_DIR, device
)

# Save model
torch.save(resnet50_transfer_learning_model.state_dict(), EXPERIMENT_3_OUTPUT_DIR / f'{EXPERIMENT_3_NAME}_Model_{TIMESTAMP}.pth')
print(f"\nModel saved: {EXPERIMENT_3_OUTPUT_DIR / f'{EXPERIMENT_3_NAME}_Model_{TIMESTAMP}.pth'}")

### Test on Unseen Images (Experiment 3)

Test the trained model on individual images from the **test set** — images that were **never seen during training**. This demonstrates whether the model has truly learned to generalise rather than memorising training data.

**Testing Options:**
- Set `RANDOM_CLASS = True` for fully random test image selection
- Set `RANDOM_CLASS = False` and specify `TARGET_CLASS` to test specific classes (useful for investigating confusion matrix patterns)

In [ ]:
# Test set statistics
print_test_set_class_statistics()

In [ ]:
# Test on unseen image from test set

RANDOM_CLASS = True
TARGET_CLASS = 'mildimpairment'  # Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'

SELECTED_TEST_IMAGE_PATH = select_random_test_image_path(
    random_class=RANDOM_CLASS,
    target_class=TARGET_CLASS if not RANDOM_CLASS else None
)

if SELECTED_TEST_IMAGE_PATH:
    resnet50_single_image_result = predict_and_visualise_single_image(
        model=resnet50_transfer_learning_model,
        image_path=SELECTED_TEST_IMAGE_PATH,
        class_names=class_names,
        device=device,
        data_dir=DATA_DIR
    )

<hr style="border:1px solid black"> </hr>

## 14. Experiment 4: Vision Transformer (ViT-B/16)

**Note:** This uses torchvision's built-in ViT-B/16 with ImageNet-1K pretrained weights, which works offline.

In [ ]:
print("EXPERIMENT 4: VISION TRANSFORMER (ViT-B/16)")

EXPERIMENT_4_NAME = "Exp4_ViT_B16_Improved"
EXPERIMENT_4_OUTPUT_DIR = OUTPUT_DIR / 'exp4'

# ViT adaptations for small medical dataset:
# - Unfreeze 8 transformer layers for better domain adaptation
# - Unfreeze patch embedding layer (critical for MRI domain shift)
# - Lower learning rate (5e-5) to prevent catastrophic forgetting

training_loader_experiment4, validation_loader_experiment4, test_loader_experiment4, _, _ = create_stratified_train_validation_test_loaders(
    DATA_DIR,
    train_transforms=baseline_transforms_no_augmentation,
    validation_and_test_transforms=validation_and_test_transforms,
    batch_size=BATCH_SIZE,
    test_split=TEST_SPLIT,
    val_split=VAL_SPLIT,
    save_splits=False,
    output_dir=OUTPUT_DIR
)

In [ ]:
# Check if ViT is available (requires sufficient GPU memory)
VIT_MODEL_AVAILABLE = True
try:
    # Create IMPROVED ViT model with more aggressive fine-tuning
    set_reproducibility_seed(42)
    vit_classifier_model = ViTClassifier(
        num_classes=NUM_CLASSES,
        fine_tune_mode='partial',
        num_unfrozen_layers=8,      # INCREASED from 4
        unfreeze_patch_embed=True,  # NEW: Critical for domain adaptation
        dropout_rate=0.3
    ).to(device)

    # Use LOWER learning rate to prevent catastrophic forgetting with more unfrozen params
    EXPERIMENT_4_LEARNING_RATE = 5e-5  # Lower than default (was 1e-4)

    vit_loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))
    vit_optimiser = optim.Adam(vit_classifier_model.parameters(), lr=EXPERIMENT_4_LEARNING_RATE)

    print(f"\nLoss: CrossEntropyLoss (with class weights)")
    print(f"Optimiser: Adam (lr={EXPERIMENT_4_LEARNING_RATE}) - lower to prevent forgetting")
    print(f"\nKey improvements over original ViT setup:")
    print(f"  - 8 unfrozen transformer layers (was 4)")
    print(f"  - Patch embedding unfrozen (was frozen)")
    print(f"  - Position embeddings unfrozen (was frozen)")
    print(f"  - Lower learning rate: {EXPERIMENT_4_LEARNING_RATE} (prevents catastrophic forgetting)")

except Exception as e:
    print(f"\n[Note] Warning: Could not initialise ViT model: {e}")
    print("This may be due to insufficient GPU memory.")
    print("Skipping Experiment 4.")
    VIT_MODEL_AVAILABLE = False

In [ ]:
if VIT_MODEL_AVAILABLE:
    # Train ViT
    vit_training_history = train_model_with_early_stopping(
        vit_classifier_model, training_loader_experiment4, validation_loader_experiment4, vit_loss_function, vit_optimiser,
        num_epochs=NUM_EPOCHS, device=device, patience=EARLY_STOPPING_PATIENCE,
        exp_name=EXPERIMENT_4_NAME, exp_dir=EXPERIMENT_4_OUTPUT_DIR
    )

In [ ]:
if VIT_MODEL_AVAILABLE:
    # Run full evaluation
    vit_metrics, vit_auc_score, vit_true_labels, vit_predicted_labels, vit_predicted_probabilities = run_complete_evaluation_with_visualisations(
        vit_classifier_model, test_loader_experiment4, class_names, EXPERIMENT_4_NAME, EXPERIMENT_4_OUTPUT_DIR, device
    )

    # Save model
    torch.save(vit_classifier_model.state_dict(), EXPERIMENT_4_OUTPUT_DIR / f'{EXPERIMENT_4_NAME}_Model_{TIMESTAMP}.pth')
    print(f"\nModel saved: {EXPERIMENT_4_OUTPUT_DIR / f'{EXPERIMENT_4_NAME}_Model_{TIMESTAMP}.pth'}")

### Test on Unseen Images (Experiment 4)

Test the trained model on individual images from the **test set** — images that were **never seen during training**. This demonstrates whether the model has truly learned to generalise rather than memorising training data.

**Testing Options:**
- Set `RANDOM_CLASS = True` for fully random test image selection
- Set `RANDOM_CLASS = False` and specify `TARGET_CLASS` to test specific classes (useful for investigating confusion matrix patterns)

In [ ]:
# Test set statistics
print_test_set_class_statistics()

In [ ]:
# Test on unseen image from test set

RANDOM_CLASS = True
TARGET_CLASS = 'mildimpairment'  # Options: 'noimpairment', 'mildimpairment', 'moderateimpairment'

if VIT_MODEL_AVAILABLE:
    SELECTED_TEST_IMAGE_PATH = select_random_test_image_path(
        random_class=RANDOM_CLASS,
        target_class=TARGET_CLASS if not RANDOM_CLASS else None
    )

    if SELECTED_TEST_IMAGE_PATH:
        vit_single_image_result = predict_and_visualise_single_image(
            model=vit_classifier_model,
            image_path=SELECTED_TEST_IMAGE_PATH,
            class_names=class_names,
            device=device,
            data_dir=DATA_DIR
        )
else:
    print("ViT model not available - skipping test")

<hr style="border:1px solid black"> </hr>

## 15. Results Comparison

Comparing all experiments to demonstrate the impact of each technique.

In [ ]:
# Compile results into comparison table
experiment_results_summary = {
    'Experiment': [
        '1. Baseline CNN (no aug)',
        '2. Baseline CNN + Aug',
        '3. ResNet50 (TL)'
    ],
    'Accuracy': [
        f"{baseline_cnn_metrics['accuracy']*100:.2f}%",
        f"{improved_baseline_metrics['accuracy']*100:.2f}%",
        f"{resnet50_metrics['accuracy']*100:.2f}%"
    ],
    'Precision': [
        f"{baseline_cnn_metrics['precision']*100:.2f}%",
        f"{improved_baseline_metrics['precision']*100:.2f}%",
        f"{resnet50_metrics['precision']*100:.2f}%"
    ],
    'Recall': [
        f"{baseline_cnn_metrics['recall']*100:.2f}%",
        f"{improved_baseline_metrics['recall']*100:.2f}%",
        f"{resnet50_metrics['recall']*100:.2f}%"
    ],
    'F1-Score': [
        f"{baseline_cnn_metrics['f1']*100:.2f}%",
        f"{improved_baseline_metrics['f1']*100:.2f}%",
        f"{resnet50_metrics['f1']*100:.2f}%"
    ],
    'AUC': [
        f"{baseline_cnn_auc_score:.4f}",
        f"{improved_baseline_auc_score:.4f}",
        f"{resnet50_auc_score:.4f}"
    ]
}

# Add ViT if available
if VIT_MODEL_AVAILABLE:
    experiment_results_summary['Experiment'].append('4. ViT (ViT)')
    experiment_results_summary['Accuracy'].append(f"{vit_metrics['accuracy']*100:.2f}%")
    experiment_results_summary['Precision'].append(f"{vit_metrics['precision']*100:.2f}%")
    experiment_results_summary['Recall'].append(f"{vit_metrics['recall']*100:.2f}%")
    experiment_results_summary['F1-Score'].append(f"{vit_metrics['f1']*100:.2f}%")
    experiment_results_summary['AUC'].append(f"{vit_auc_score:.4f}")

experiment_results_dataframe = pd.DataFrame(experiment_results_summary)

print("RESULTS COMPARISON")
print(experiment_results_dataframe.to_string(index=False))

In [ ]:
def plot_and_save_final_experiment_comparison(experiment_results_dataframe, output_dir):
    """Create final comparison plots."""
    fig = plt.figure(figsize=(16, 14))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.25)

    experiment_names = experiment_results_dataframe['Experiment'].tolist()
    accuracy_values = [float(x.replace('%', '')) for x in experiment_results_dataframe['Accuracy']]
    f1_score_values = [float(x.replace('%', '')) for x in experiment_results_dataframe['F1-Score']]
    auc_values = [float(x) for x in experiment_results_dataframe['AUC']]
    precision_values = [float(x.replace('%', '')) for x in experiment_results_dataframe['Precision']]
    recall_values = [float(x.replace('%', '')) for x in experiment_results_dataframe['Recall']]

    bar_colours = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'][:len(experiment_names)]

    # Accuracy comparison
    ax1 = fig.add_subplot(gs[0, 0])
    accuracy_bars = ax1.bar(experiment_names, accuracy_values, color=bar_colours, edgecolor='black')
    ax1.set_ylabel('Accuracy (%)', fontsize=12)
    ax1.set_title('Accuracy Comparison', fontweight='bold', fontsize=14)
    ax1.set_ylim(0, 100)
    for bar_patch, acc in zip(accuracy_bars, accuracy_values):
        ax1.text(bar_patch.get_x() + bar_patch.get_width()/2, bar_patch.get_height() + 1,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    # Rotate x-axis labels to prevent overlap
    plt.sca(ax1)
    plt.xticks(rotation=45, ha='right', fontsize=10)

    # F1-Score comparison
    ax2 = fig.add_subplot(gs[0, 1])
    f1_bars = ax2.bar(experiment_names, f1_score_values, color=bar_colours, edgecolor='black')
    ax2.set_ylabel('F1-Score (%)', fontsize=12)
    ax2.set_title('F1-Score Comparison', fontweight='bold', fontsize=14)
    ax2.set_ylim(0, 100)
    for bar_patch, f1 in zip(f1_bars, f1_score_values):
        ax2.text(bar_patch.get_x() + bar_patch.get_width()/2, bar_patch.get_height() + 1,
                 f'{f1:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    # Rotate x-axis labels to prevent overlap
    plt.sca(ax2)
    plt.xticks(rotation=45, ha='right', fontsize=10)

    # AUC comparison
    ax3 = fig.add_subplot(gs[1, 0])
    auc_bars = ax3.bar(experiment_names, auc_values, color=bar_colours, edgecolor='black')
    ax3.set_ylabel('AUC Score', fontsize=12)
    ax3.set_title('AUC Comparison', fontweight='bold', fontsize=14)
    ax3.set_ylim(0, 1.0)
    ax3.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    for bar_patch, auc_val in zip(auc_bars, auc_values):
        ax3.text(bar_patch.get_x() + bar_patch.get_width()/2, bar_patch.get_height() + 0.02,
                 f'{auc_val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    # Rotate x-axis labels to prevent overlap
    plt.sca(ax3)
    plt.xticks(rotation=45, ha='right', fontsize=10)

    # Precision vs Recall
    ax4 = fig.add_subplot(gs[1, 1])
    x_positions = np.arange(len(experiment_names))
    width = 0.35
    precision_bars = ax4.bar(x_positions - width/2, precision_values, width, label='Precision', color='#3498db', edgecolor='black')
    recall_bars = ax4.bar(x_positions + width/2, recall_values, width, label='Recall', color='#e74c3c', edgecolor='black')
    ax4.set_ylabel('Score (%)', fontsize=12)
    ax4.set_title('Precision vs Recall', fontweight='bold', fontsize=14)
    ax4.set_xticks(x_positions)
    ax4.set_ylim(0, 100)
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    # Rotate x-axis labels to prevent overlap
    ax4.set_xticklabels(experiment_names, rotation=45, ha='right', fontsize=10)

    # Add main title
    fig.suptitle('Experiment Results Comparison\nBrain MRI Classification for Dementia Severity',
                 fontweight='bold', fontsize=16, y=0.98)

    # Adjusted layout to accommodate rotated labels
    plt.tight_layout(rect=[0, 0.02, 1, 0.95])

    save_path = output_dir / f'Experiment_Comparison_{TIMESTAMP}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

plot_and_save_final_experiment_comparison(experiment_results_dataframe, OUTPUT_DIR)